<a href="https://colab.research.google.com/github/Murcha1990/ML_AI26/blob/main/Week3_GD_Features/Regularizations_Encoding_Pro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Неклассические регуляризации

### **Elastic Net**

**Elastic Net** — это метод регуляризации, который комбинирует $\ell_1$ (LASSO) и $\ell_2$ (Ridge) регуляризации через линейное взвешивание.  

Формула Elastic Net:
$$
R_{\text{Elastic Net}}(\mathbf{w}) = \alpha \|\mathbf{w}\|_1 + (1 - \alpha) \frac{1}{2} \|\mathbf{w}\|_2^2,
$$
где:
- $\|\mathbf{w}\|_1 = \sum |w_i|$ — $\ell_1$-норма, отвечающая за разреженность (спарсити);
- $\|\mathbf{w}\|_2^2 = \sum w_i^2$ — $\ell_2$-норма, обеспечивающая устойчивость к шуму;
- $\alpha \in [0, 1]$ — гиперпараметр, задающий баланс между $\ell_1$- и $\ell_2$-нормами.

В sklearn баланс задаётся параметром `l1_ratio`, а `alpha` — это общая сила регуляризации: $\text{alpha}\cdot\left(\text{l1\_ratio}\,\|\mathbf{w}\|_1 + \frac{1-\text{l1\_ratio}}{2}\|\mathbf{w}\|_2^2\right)$.

#### **Основные свойства Elastic Net:**
1. **Смешивание разреженности и устойчивости:**
   - $\ell_1$-норма отбирает признаки, зануляя неважные веса.
   - $\ell_2$-норма уменьшает веса всех признаков, предотвращая нестабильность и переобучение.
   
2. **Скалируемость:**
   - Elastic Net хорошо работает с высокоразмерными данными, особенно в случаях, когда признаки сильно коррелированы.

3. **Простая оптимизация:**
   - Функция Elastic Net является выпуклой, что упрощает её минимизацию.

---

### **Huber-подобная регуляризация**

**Huber-подобная регуляризация** также объединяет $\ell_1$- и $\ell_2$ -свойства, но делает это адаптивно, в зависимости от величины весов:

Формула:
$$
R_{\delta}(\mathbf{w}) = \sum_{i=1}^n
\begin{cases}
\frac{1}{2} w_i^2 & \text{если } |w_i| \leq \delta, \\
\delta (|w_i| - \frac{1}{2} \delta) & \text{если } |w_i| > \delta.
\end{cases}
$$

#### **Основные свойства Huber-регуляризации:**
1. **Адаптивность:**
   - Для малых весов $(|w_i| \leq \delta)$ штраф квадратичный, как в $\ell_2$: веса плавно уменьшаются, но **не зануляются** (в отличие от $\ell_1$).
   - Для больших весов $(|w_i| > \delta)$ штраф растёт линейно, как в $\ell_1$: большие (важные) веса сжимаются слабее, чем при $\ell_2$-регуляризации.

2. **Выпуклость и гладкость:**
   - Функция Хьюбера выпуклая и дифференцируемая, поэтому задача легко решается градиентными методами.

3. **Важно не путать с Huber-регрессией:**
   - Регуляризация штрафует **веса** модели и не делает её устойчивой к выбросам в данных.
   - Устойчивость к выбросам даёт **функция потерь** Хьюбера (`HuberRegressor`, см. ниже).

---

### **Ключевые различия**

| **Характеристика**               | **Elastic Net**                                            | **Huber-подобная регуляризация**                          |
|-----------------------------------|-----------------------------------------------------------|----------------------------------------------------------|
| **Баланс $\ell_1$ и $\ell_2$** | Линейная комбинация двух норм                 | Адаптивное переключение между $\ell_1$ и $\ell_2$    |
| **Цель**                          | Обеспечить разреженность и устойчивость                   | Сжимать малые веса как $\ell_2$, не пережимая большие      |
| **Способ применения**             | Все признаки сразу штрафуются линейной комбинацией норм   | Штраф зависит от значения весов $|w_i|$              |
| **Параметры**                     | Сила регуляризации и баланс $\ell_1$/$\ell_2$ (`alpha` и `l1_ratio` в sklearn) | Сила регуляризации и порог переключения $\delta$         |
| **Выпуклость функции**            | Выпуклая (легко оптимизируется)                          | Выпуклая и гладкая (легко оптимизируется)                |
| **Отбор признаков**               | Да: $\ell_1$-часть зануляет веса                          | Нет: веса уменьшаются, но не зануляются                   |
| **Применение**                    | Высокая корреляция признаков, отбор признаков             | Когда есть и много мелких, и несколько крупных важных весов |

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error
from itertools import combinations
from math import log

# 1. Генерация данных для многомерной линейной регрессии
np.random.seed(42)
n_samples, n_features, n_targets = 100, 5, 1

X = np.random.rand(n_samples, n_features)
true_coefs = np.array([
    [3, 0, 0, 2, 0],  # Для первой компоненты Y
    [0, -1, 0, 0, 1],  # Для второй компоненты Y
    [2, 0, 1, 0, 0]    # Для третьей компоненты Y
])

Y = X @ true_coefs.T + np.random.normal(scale=1.0, size=(n_samples, n_targets))

# 2. Добавление выбросов (5% от общего числа наблюдений)
n_outliers = int(0.05 * n_samples)
outlier_indices = np.random.choice(np.arange(n_samples), n_outliers, replace=False)

for i in outlier_indices:
    Y[i] += np.random.normal(scale=200, size=n_targets)  # Добавляем большие случайные значения

# Преобразование данных в DataFrame для удобства
df = pd.DataFrame(X, columns=[f"Feature_{i}" for i in range(n_features)])
for i in range(n_targets):
    df[f"Target_{i}"] = Y[:, i]

In [ ]:
df.head()

,Feature_0,Feature_1,Feature_2,Feature_3,Feature_4,Target_0
0,0.374540,0.950714,0.731994,0.598658,0.156019,2.662693
1,0.155995,0.058084,0.866176,0.601115,0.708073,3.546384
2,0.020584,0.969910,0.832443,0.212339,0.181825,1.436856
3,0.183405,0.304242,0.524756,0.431945,0.291229,0.837200
4,0.611853,0.139494,0.292145,0.366362,0.456070,1.669868


In [ ]:
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import train_test_split

# 2. Разделение данных на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(df.drop(['Target_0'], axis=1), df['Target_0'], test_size=0.2, random_state=42)

# 3. Elastic Net
elastic_net = ElasticNet(alpha=0.5, l1_ratio=0.5, random_state=42)  # l1_ratio: баланс L1 и L2
elastic_net.fit(X_train, y_train)

# Предсказание и оценка
y_pred_en = elastic_net.predict(X_test)
mse_en = mean_squared_error(y_test, y_pred_en)

print(f"Mean Squared Error (Elastic Net): {mse_en:.3f}")

Mean Squared Error (Elastic Net): 8151.304


In [ ]:
from scipy.optimize import minimize
from sklearn.base import BaseEstimator, RegressorMixin

# 2. Функция Хьюбера для весов и её производная
def huber_penalty(w, delta=1.0):
    a = np.abs(w)
    return np.sum(np.where(a <= delta, 0.5 * w**2, delta * (a - 0.5 * delta)))

def huber_penalty_grad(w, delta=1.0):
    return np.clip(w, -delta, delta)   # w при |w| <= delta, ±delta иначе

# 3. Линейная регрессия: минимизируем MSE + alpha * R_delta(w)
class HuberRegularizedRegression(BaseEstimator, RegressorMixin):
    def __init__(self, alpha=1.0, delta=1.0):
        self.alpha = alpha
        self.delta = delta

    def fit(self, X, y):
        X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
        n, d = X.shape

        def loss_and_grad(params):
            w, b = params[:-1], params[-1]          # свободный член не штрафуем
            residual = X @ w + b - y
            loss = np.mean(residual**2) + self.alpha * huber_penalty(w, self.delta)
            grad_w = 2 / n * X.T @ residual + self.alpha * huber_penalty_grad(w, self.delta)
            grad_b = 2 / n * residual.sum()
            return loss, np.append(grad_w, grad_b)

        res = minimize(loss_and_grad, np.zeros(d + 1), jac=True, method='L-BFGS-B')
        self.coef_, self.intercept_ = res.x[:-1], res.x[-1]
        return self

    def predict(self, X):
        return np.asarray(X, dtype=float) @ self.coef_ + self.intercept_

# Обучаем модель с Huber-регуляризацией
model = HuberRegularizedRegression(alpha=0.01, delta=0.5)
model.fit(X_train, y_train)

# 4. Оценка модели на тестовых данных
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)

print(f"Test MSE with Huber-like regularization: {mse:.4f}")
print("Веса:", model.coef_.round(3))

Есть еще huber-регрессия, но это не регуляризация!

In [ ]:
from sklearn.linear_model import HuberRegressor

# 4. Huber-регрессия
huber = HuberRegressor(alpha=0.5, max_iter=1000)
huber.fit(X_train, y_train)

# Предсказание и оценка
y_pred_huber = huber.predict(X_test)
mse_huber = mean_squared_error(y_test, y_pred_huber)

print(f"Mean Squared Error (Huber): {mse_huber:.3f}")

Mean Squared Error (Huber): 8080.887


# Подбор гиперпараметров по кросс-валидации

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {'alpha' : [0.01, 0.1, 0.5, 1, 10]}

model = HuberRegressor(max_iter=1000)

gs = GridSearchCV(model, params, cv=3, scoring='neg_mean_squared_error')
gs.fit(X_train, y_train)

GridSearchCV(cv=3, estimator=HuberRegressor(max_iter=1000),
             param_grid={'alpha': [0.01, 0.1, 0.5, 1, 10]},
             scoring='neg_mean_squared_error')

In [ ]:
-gs.best_score_, gs.best_estimator_, gs.best_params_

(3166.866866716891, HuberRegressor(alpha=10, max_iter=1000), {'alpha': 10})

In [ ]:
y_pred_huber_tuned = gs.predict(X_test)
mse_huber_tuned = mean_squared_error(y_test, y_pred_huber_tuned)

print(f"Mean Squared Error (Huber, tuned): {mse_huber_tuned:.3f}")

Mean Squared Error (Huber, tuned): 8049.108


# Информационные критерии

**AIC** (информационный критерий Акаике) и **BIC** (байесовский информационный критерий) — это два статистических критерия, которые используются для выбора модели среди множества кандидатов. Оба критерия оценивают баланс между точностью модели (ее способностью подгонять данные) и сложностью модели (числом параметров).

Они являются **критериями информации**, что означает, что их задача — минимизировать потерю информации при выборе модели. В идеале мы хотим выбрать такую модель, которая точно описывает данные, но не является излишне сложной.

---

### 1. **AIC (Информационный критерий Акаике)**

**Формула:**

$$
\text{AIC} = -2 \ln(L) + 2k
$$

Где:
- $ L $ — функция правдоподобия модели (то есть вероятность наблюдаемых данных при условии выбранной модели).
- $ k $ — количество параметров в модели (например, количество коэффициентов в линейной регрессии).
- $ \ln $ — натуральный логарифм.

**Интерпретация AIC:**
- **Меньшее значение AIC** означает лучшую модель, так как оно отражает баланс между хорошей подгонкой данных (чем выше правдоподобие $ L $, тем меньше AIC) и простотой модели (чем больше параметров, тем больше значение AIC).
- AIC штрафует модели с большим количеством параметров, тем самым предотвращая переобучение.

**Использование AIC:**
- AIC используется для выбора между несколькими моделями, которые описывают одни и те же данные. Модель с наименьшим значением AIC считается наиболее оптимальной, так как она балансирует точность и сложность.

---

### 2. **BIC (Байесовский информационный критерий)**

**Формула:**

$$
\text{BIC} = -2 \ln(L) + k \ln(n)
$$

Где:
- $ L $ — функция правдоподобия модели (также, как и в AIC).
- $ k $ — количество параметров модели.
- $ n $ — количество наблюдений (размер выборки).

**Интерпретация BIC:**
- **BIC** схож с AIC, но включает в себе дополнительный фактор — размер выборки $ n $. Этот фактор увеличивает штраф за сложность модели с ростом размера выборки. Таким образом, BIC более строго штрафует за добавление лишних параметров, чем AIC, особенно при большом объеме данных.
- Меньшее значение BIC указывает на более подходящую модель.

**Использование BIC:**
- BIC также используется для выбора лучшей модели среди нескольких. Однако, в отличие от AIC, BIC склонен отдавать предпочтение более простым моделям, особенно если количество наблюдений велико, поскольку $ \ln(n) $ растет с увеличением размера выборки.

In [ ]:
# Преобразование данных в DataFrame для удобства
df_train = pd.DataFrame(X_train, columns=[f"Feature_{i}" for i in range(n_features)])
for i in range(n_targets):
    df_train[f"Target_{i}"] = y_train

# 2. Функция для расчета AIC и BIC для многомерной регрессии
def calculate_aic_bic_multi(model, X, Y):
    n, k = X.shape[0], X.shape[1]  # Число наблюдений и признаков
    n_targets = Y.shape[1]  # Число целевых переменных
    y_pred = model.predict(X)
    residual_sum_of_squares = np.sum((Y - y_pred) ** 2)

    # Средняя ошибка для всех компонент целевой переменной
    mse = mean_squared_error(Y, y_pred, multioutput='uniform_average')
    log_likelihood = -n * n_targets / 2 * (np.log(2 * np.pi * mse) + 1)

    # Количество параметров = (число признаков + 1) * число целевых переменных
    total_params = n_targets * (k + 1)

    aic = 2 * total_params - 2 * log_likelihood
    bic = total_params * log(n) - 2 * log_likelihood

    return aic, bic

# 3. Перебор комбинаций признаков для минимизации AIC/BIC
min_aic, min_bic = float('inf'), float('inf')
best_aic_model, best_bic_model = None, None

for num_features in range(1, n_features + 1):
    for feature_subset in combinations(df_train.columns[:n_features], num_features):
        # Подготовка данных
        X_subset = df_train[list(feature_subset)].values
        Y_subset = df_train[[f"Target_{i}" for i in range(n_targets)]].values

        # Обучение модели
        model = LinearRegression()
        model.fit(X_subset, Y_subset)

        # Расчет AIC и BIC
        aic, bic = calculate_aic_bic_multi(model, X_subset, Y_subset)

        # Обновление лучших моделей
        if aic < min_aic:
            min_aic = aic
            best_aic_model = (model, feature_subset)

        if bic < min_bic:
            min_bic = bic
            best_bic_model = (model, feature_subset)

# 4. Вывод результатов
print(f"Best AIC: {min_aic:.2f} using features {best_aic_model[1]}")
print(f"Best BIC: {min_bic:.2f} using features {best_bic_model[1]}")

# 5. Оценка на тестовой выборке
X_test_np = X_test.values

# Access columns by index using NumPy-style slicing
X_test_aic = X_test_np[:, [int(feature.split('_')[1]) for feature in best_aic_model[1]]]
X_test_bic = X_test_np[:, [int(feature.split('_')[1]) for feature in best_bic_model[1]]]

aic_test_mse = mean_squared_error(y_test, best_aic_model[0].predict(X_test_aic), multioutput='uniform_average')
bic_test_mse = mean_squared_error(y_test, best_bic_model[0].predict(X_test_bic), multioutput='uniform_average')

print(f"Test MSE for AIC-selected model: {aic_test_mse:.4f}")
print(f"Test MSE for BIC-selected model: {bic_test_mse:.4f}")

Best AIC: 874.91 using features ('Feature_2',)
Best BIC: 879.68 using features ('Feature_2',)
Test MSE for AIC-selected model: 7749.0920
Test MSE for BIC-selected model: 7749.0920


# Кодирование категориальных признаков: сравнение подходов

Сравним на одной задаче все способы кодирования, которые мы обсуждали:

- One-Hot Encoding
- Mean target encoding (счётчики со сглаживанием, посчитанные на кросс-валидации)
- **Expanding mean**
- **Хэширование признаков**

Задача регрессии: по характеристикам бриллианта предсказываем его цену. Категориальные признаки — `cut`, `color`, `clarity`.

In [ ]:
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder
from sklearn.feature_extraction import FeatureHasher
from sklearn.model_selection import KFold, cross_val_score

diamonds = sns.load_dataset('diamonds')

X_d = diamonds.drop(columns='price')
y_d = diamonds['price'].astype(float)

num_cols = ['carat', 'depth', 'table', 'x', 'y', 'z']
cat_cols = ['cut', 'color', 'clarity']

X_d[cat_cols] = X_d[cat_cols].astype(str)
X_d.head()

Чтобы разница между кодировками была заметнее, добавим признак с большим числом категорий — сочетание всех трёх категориальных признаков. В нём около 270 разных значений: One-Hot даст столько же столбцов, а счётчикам и хэшированию это не страшно.

In [ ]:
X_d['cut_color_clarity'] = X_d['cut'] + '_' + X_d['color'] + '_' + X_d['clarity']
cat_cols_big = cat_cols + ['cut_color_clarity']

X_d[cat_cols_big].nunique()

## Expanding mean

Для объекта с номером $m$ код считается только по объектам **до него**:

$$\text{code}_m = \frac{\sum_{i<m,\ c_i = c_m} y_i + \alpha \cdot prior}{n_{<m}(c_m) + \alpha}$$

Датасет `diamonds` отсортирован по цене, поэтому перед подсчётом обязательно перемешиваем строки. Иначе каждый следующий объект будет дороже всех предыдущих, и код систематически занизит цену.

- `fit_transform` (на трейне) считает expanding mean по случайной перестановке. Можно усреднить по нескольким перестановкам — так делает CatBoost.
- `transform` (на новых данных) использует сглаженные средние по всей обучающей выборке.

In [ ]:
class ExpandingMeanEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, alpha=10, n_permutations=1, random_state=42):
        self.alpha = alpha
        self.n_permutations = n_permutations
        self.random_state = random_state

    def fit(self, X, y):
        X = pd.DataFrame(X).reset_index(drop=True)
        y = pd.Series(np.asarray(y, dtype=float))
        self.prior_ = y.mean()
        self.codes_ = {}
        for col in X.columns:
            stats = y.groupby(X[col]).agg(['sum', 'count'])
            self.codes_[col] = (stats['sum'] + self.alpha * self.prior_) / (stats['count'] + self.alpha)
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        return np.column_stack([X[col].map(self.codes_[col]).fillna(self.prior_).astype(float)
                                for col in X.columns])

    def fit_transform(self, X, y):
        self.fit(X, y)
        X = pd.DataFrame(X).reset_index(drop=True)
        y = pd.Series(np.asarray(y, dtype=float))
        rng = np.random.RandomState(self.random_state)
        result = np.zeros((len(X), X.shape[1]))

        for _ in range(self.n_permutations):
            perm = rng.permutation(len(X))
            Xp, yp = X.iloc[perm].reset_index(drop=True), y.iloc[perm].reset_index(drop=True)
            for j, col in enumerate(X.columns):
                grp = yp.groupby(Xp[col])
                sum_before = grp.cumsum() - yp      # сумма таргета ДО объекта
                cnt_before = grp.cumcount()         # число объектов ДО него
                codes = (sum_before + self.alpha * self.prior_) / (cnt_before + self.alpha)
                result[perm, j] += codes.values
        return result / self.n_permutations

In [ ]:
# пример: expanding mean для признака cut на первых 8 объектах
em = ExpandingMeanEncoder(alpha=1)
pd.DataFrame({'cut': X_d['cut'][:8].values,
              'price': y_d[:8].values,
              'expanding mean': em.fit_transform(X_d[['cut']][:8], y_d[:8])[:, 0].round(1)})

## Хэширование

Каждое значение `признак=категория` хэшируем в одну из `n_features` корзин. Словарь категорий хранить не нужно. Если корзин мало, разные категории сливаются (коллизии).

In [ ]:
class HashingEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, n_features=2**8):
        self.n_features = n_features

    def fit(self, X, y=None):
        return self   # учить нечего

    def transform(self, X):
        X = pd.DataFrame(X)
        tokens = [[f'{col}={val}' for col, val in zip(X.columns, row)] for row in X.itertuples(index=False)]
        hasher = FeatureHasher(n_features=self.n_features, input_type='string', alternate_sign=False)
        return hasher.transform(tokens)

In [ ]:
# сколько разных значений слиплось в одну корзину при разном числе корзин
values = [f'cut_color_clarity={v}' for v in X_d['cut_color_clarity'].unique()]
for n in [2**4, 2**6, 2**8, 2**12]:
    buckets = FeatureHasher(n_features=n, input_type='string').transform([[v] for v in values]).indices
    print(f'{n:5d} корзин: {len(values)} значений -> {len(set(buckets))} занятых корзин')

## Сравнение всех подходов

Собираем пайплайны: масштабирование числовых признаков + кодирование категориальных + Ridge-регрессия. Качество меряем по RMSE на 5-fold кросс-валидации (чем меньше, тем лучше).

In [ ]:
def make_pipe(encoder, cats):
    prep = ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', encoder, cats),
    ])
    return Pipeline([('prep', prep), ('model', Ridge(alpha=1.0))])

encoders = {
    'Только числовые':        'drop',
    'One-Hot':                OneHotEncoder(handle_unknown='ignore'),
    'Target encoding (CV)':   TargetEncoder(target_type='continuous'),
    'Expanding mean':         ExpandingMeanEncoder(alpha=10),
    'Expanding mean ×5':      ExpandingMeanEncoder(alpha=10, n_permutations=5),
    'Hashing, 16 корзин':     HashingEncoder(n_features=2**4),
    'Hashing, 256 корзин':    HashingEncoder(n_features=2**8),
    'Hashing, 4096 корзин':   HashingEncoder(n_features=2**12),
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

for cats_name, cats in [('3 признака', cat_cols), ('+ сочетание (≈270 значений)', cat_cols_big)]:
    for name, enc in encoders.items():
        rmse = -cross_val_score(make_pipe(enc, cats), X_d, y_d, cv=cv,
                                scoring='neg_root_mean_squared_error')
        results.append({'категориальные': cats_name, 'кодирование': name, 'RMSE': rmse.mean().round(1)})

table = pd.DataFrame(results).pivot(index='кодирование', columns='категориальные', values='RMSE')
table.loc[list(encoders), ['3 признака', '+ сочетание (≈270 значений)']]

**Что видно из сравнения**

- Любое кодирование лучше, чем просто выбросить категориальные признаки: RMSE падает с ~1500 до ~1120–1290.
- Для **линейной** модели лучше всего One-Hot: каждая категория получает свой вес. Счётчики сжимают категорию в одно число, и линейной модели этого мало, тем более что средняя цена категории перемешана с размером камня. Сильнее всего счётчики раскрываются в деревьях и бустинге — например, в CatBoost.
- Target encoding на кросс-валидации и expanding mean дают почти одинаковое качество. Усреднение expanding mean по нескольким перестановкам немного снижает шум.
- Хэширование с 4096 корзинами почти не даёт коллизий и работает как One-Hot. Если корзин всего 16, разные категории сливаются и качество заметно падает, особенно для признака с ≈270 значениями.

# Хэширование текстов

Самый частый пример хэширования — тексты. Обычный мешок слов (`CountVectorizer`) сначала строит **словарь** всех слов корпуса и отводит каждому слову свой столбец. На больших корпусах словарь занимает миллионы слов, а новые слова, которых не было при обучении, просто теряются.

`HashingVectorizer` словарь не хранит: номер столбца для слова — это хэш слова по модулю `n_features`.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, HashingVectorizer

train_texts = ['cheap diamond with good cut', 'expensive diamond ideal cut', 'ring with small diamond']
new_texts = ['expensive ring with sapphire']   # слова sapphire не было в обучении

cv_text = CountVectorizer().fit(train_texts)
hv = HashingVectorizer(n_features=2**10, alternate_sign=False, norm=None)

print('Словарь CountVectorizer:', len(cv_text.vocabulary_), 'слов')
print('CountVectorizer   -> ненулевых признаков в новом тексте:', cv_text.transform(new_texts).nnz)
print('HashingVectorizer -> ненулевых признаков в новом тексте:', hv.transform(new_texts).nnz, '(fit не нужен)')

## Классификация текстов

Возьмём два раздела новостного датасета 20 Newsgroups — про космос и про автомобили — и научим логистическую регрессию отличать один от другого. Сравним кодирование через словарь и через хэширование с разным числом корзин.

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import time

cats = ['sci.space', 'rec.autos']
train = fetch_20newsgroups(subset='train', categories=cats, remove=('headers', 'footers', 'quotes'))
test = fetch_20newsgroups(subset='test', categories=cats, remove=('headers', 'footers', 'quotes'))

print(len(train.data), 'текстов в обучении,', len(test.data), 'в тесте')
print(train.target_names)
print(train.data[0][:300])

In [ ]:
vectorizers = {
    'CountVectorizer (словарь)': CountVectorizer(),
    'Hashing, 2^18 корзин':      HashingVectorizer(n_features=2**18, alternate_sign=False),
    'Hashing, 2^10 корзин':      HashingVectorizer(n_features=2**10, alternate_sign=False),
    'Hashing, 2^6 корзин':       HashingVectorizer(n_features=2**6, alternate_sign=False),
}

rows = []
for name, vec in vectorizers.items():
    start = time.time()
    X_tr = vec.fit_transform(train.data)
    X_te = vec.transform(test.data)
    clf = LogisticRegression(max_iter=1000).fit(X_tr, train.target)
    rows.append({'кодирование': name,
                 'число признаков': X_tr.shape[1],
                 'accuracy': round(accuracy_score(test.target, clf.predict(X_te)), 3),
                 'время, с': round(time.time() - start, 2)})

pd.DataFrame(rows)

Что видно:

- При большом числе корзин ($2^{18}$) хэширование работает так же, как словарь: коллизий почти нет.
- При малом числе корзин много разных слов попадает в один столбец, и качество падает.
- У `CountVectorizer` число признаков равно размеру словаря, его нужно хранить вместе с моделью. У хэширования размерность задаём сами, а хранить ничего не нужно.

## Обучение на потоке данных

Поскольку `HashingVectorizer` не хранит состояние, тексты можно кодировать **по частям** — например, когда данные приходят потоком или не помещаются в память. Все батчи гарантированно получают одинаковые столбцы.

Логистическую регрессию, которая учится по батчам, даёт `SGDClassifier(loss='log_loss')` с методом `partial_fit`. С `CountVectorizer` так не получится: словарь нужно построить заранее по всем данным.

In [ ]:
from sklearn.linear_model import SGDClassifier

hv_stream = HashingVectorizer(n_features=2**18, alternate_sign=False)
sgd = SGDClassifier(loss='log_loss', random_state=42)    # логистическая регрессия, обучаемая по батчам

batch_size = 200
for start in range(0, len(train.data), batch_size):
    X_batch = hv_stream.transform(train.data[start:start + batch_size])
    y_batch = train.target[start:start + batch_size]
    sgd.partial_fit(X_batch, y_batch, classes=[0, 1])

    acc = accuracy_score(test.target, sgd.predict(hv_stream.transform(test.data)))
    print(f'обработано {min(start + batch_size, len(train.data)):4d} текстов -> accuracy на тесте {acc:.3f}')

**Итого**

- `CountVectorizer`: понятные признаки (одно слово — один столбец), но нужен словарь, а новые слова теряются.
- `HashingVectorizer`: фиксированная размерность, нет словаря, новые слова тоже кодируются, можно учиться на потоке. Плата — коллизии и неинтерпретируемые столбцы.
- Обычно берут $2^{18}$–$2^{20}$ корзин: коллизий мало, а разреженная матрица всё равно занимает немного памяти.